In [1]:
# ============================================================
# 📦 Netflix Churn Prediction — XGBoost PIPELINE
# Production-grade • Fairness-ready • InferStream-compatible
# ============================================================

import pandas as pd
import numpy as np
import joblib
import os
import json

from xgboost import XGBClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.pipeline import Pipeline

from datetime import datetime

# ================================
# Config
# ================================
DATA_PATH = "../data/netflix_customer_churn.csv"
MODEL_DIR = "../backend/models/netflix"

PIPELINE_NAME = "xgb_pipeline.pkl"
SCALER_NAME = "scaler.pkl"
ENCODER_NAME = "encoder.pkl"
FEATURES_NAME = "feature_names_xgb.json"

os.makedirs(MODEL_DIR, exist_ok=True)

# ================================
# Load Data
# ================================
df = pd.read_csv(DATA_PATH)
print(f"✅ Loaded dataset: {df.shape}")

TARGET_COL = "churned"

df = df.dropna(subset=[TARGET_COL])
df[TARGET_COL] = df[TARGET_COL].astype(int)

# Drop IDs if present
for col in ["customer_id", "customerid", "user_id"]:
    if col in df.columns:
        df = df.drop(columns=[col])

y = df[TARGET_COL]
X = df.drop(columns=[TARGET_COL])

# ================================
# Feature Groups
# ================================
num_cols = X.select_dtypes(include=["int64", "float64"]).columns.tolist()
cat_cols = X.select_dtypes(include=["object", "category"]).columns.tolist()

print("🔢 Numeric features:", num_cols)
print("🏷️ Categorical features:", cat_cols)

# ================================
# Preprocessing Pipeline
# ================================
preprocessor = ColumnTransformer(
    transformers=[
        ("num", StandardScaler(), num_cols),
        ("cat", OneHotEncoder(handle_unknown="ignore"), cat_cols),
    ]
)

# ================================
# Model
# ================================
model = XGBClassifier(
    n_estimators=400,
    max_depth=6,
    learning_rate=0.05,
    subsample=0.8,
    colsample_bytree=0.8,
    objective="binary:logistic",
    eval_metric="logloss",
    random_state=42,
    n_jobs=-1,
)

pipeline = Pipeline([
    ("preprocessor", preprocessor),
    ("model", model),
])

# ================================
# Train / Test Split
# ================================
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

print(f"🧪 Train size: {X_train.shape}")
print(f"🧪 Test size:  {X_test.shape}")

# ================================
# Train
# ================================
pipeline.fit(X_train, y_train)

# ================================
# Evaluate
# ================================
preds = pipeline.predict(X_test)

print("📊 Classification Report")
print(classification_report(y_test, preds, digits=4))

print("🧩 Confusion Matrix")
print(confusion_matrix(y_test, preds))

# ================================
# Save Artifacts (CRITICAL)
# ================================
joblib.dump(pipeline, os.path.join(MODEL_DIR, PIPELINE_NAME))

joblib.dump(
    pipeline.named_steps["preprocessor"].named_transformers_["num"],
    os.path.join(MODEL_DIR, SCALER_NAME)
)

joblib.dump(
    pipeline.named_steps["preprocessor"].named_transformers_["cat"],
    os.path.join(MODEL_DIR, ENCODER_NAME)
)

# Save feature metadata
feature_names = (
    pipeline.named_steps["preprocessor"]
    .get_feature_names_out()
    .tolist()
)

with open(os.path.join(MODEL_DIR, FEATURES_NAME), "w") as f:
    json.dump(feature_names, f, indent=2)

print("✅ Saved artifacts:")
print(f"   • {PIPELINE_NAME}")
print(f"   • {SCALER_NAME}")
print(f"   • {ENCODER_NAME}")
print(f"   • {FEATURES_NAME}")

print("🏁 Done at", datetime.now().strftime("%Y-%m-%d %H:%M:%S"))

✅ Loaded dataset: (5000, 14)
🔢 Numeric features: ['age', 'watch_hours', 'last_login_days', 'monthly_fee', 'number_of_profiles', 'avg_watch_time_per_day']
🏷️ Categorical features: ['gender', 'subscription_type', 'region', 'device', 'payment_method', 'favorite_genre']
🧪 Train size: (4000, 12)
🧪 Test size:  (1000, 12)


/var/folders/91/syqbgxdn69n01td9pvbx3dz80000gn/T/ipykernel_88047/2936162654.py:57: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  cat_cols = X.select_dtypes(include=["object", "category"]).columns.tolist()


📊 Classification Report
              precision    recall  f1-score   support

           0     0.9920    0.9980    0.9950       497
           1     0.9980    0.9920    0.9950       503

    accuracy                         0.9950      1000
   macro avg     0.9950    0.9950    0.9950      1000
weighted avg     0.9950    0.9950    0.9950      1000

🧩 Confusion Matrix
[[496   1]
 [  4 499]]
✅ Saved artifacts:
   • xgb_pipeline.pkl
   • scaler.pkl
   • encoder.pkl
   • feature_names_xgb.json
🏁 Done at 2026-01-24 18:42:02


In [5]:
import json

with open("../backend/models/netflix/feature_names_xgb.json") as f:
    FEATURE_NAMES = json.load(f)

print(len(FEATURE_NAMES))
print(FEATURE_NAMES[:20])

35
['num__age', 'num__watch_hours', 'num__last_login_days', 'num__monthly_fee', 'num__number_of_profiles', 'num__avg_watch_time_per_day', 'cat__gender_Female', 'cat__gender_Male', 'cat__gender_Other', 'cat__subscription_type_Basic', 'cat__subscription_type_Premium', 'cat__subscription_type_Standard', 'cat__region_Africa', 'cat__region_Asia', 'cat__region_Europe', 'cat__region_North America', 'cat__region_Oceania', 'cat__region_South America', 'cat__device_Desktop', 'cat__device_Laptop']
